# Part 5B: Dynamic Continuous Batching

In Part 5A, we built the foundations of continuous batching.

We started with the request lifecycle:

```text
waiting → running → finished
```

and changed generation into an iteration-level serving loop:

```text
schedule
   ↓
execute one iteration
   ↓
update requests
   ↓
schedule again
   ↓
...
```

This allowed waiting requests to enter as soon as capacity became available.

For example:

```text
Step 1: [A, B]
Step 2: [A, B]
Step 3: [A, B]

A finishes

Step 4: [B, C]
```

We then replaced simulated generation with real model inference and introduced:

- per-request KV caches
- prefill and decode
- attention masks
- padding for different sequence lengths
- one real batched model forward pass

By the end of Part 5A, we could take:

```text
running = [A, B]
```

and execute both requests through one model call:

```text
A ─┐
   ├── MODEL
B ─┘
```

However, that batch was still effectively static.

The harder problem begins when the active batch changes while generation is already in progress.

Suppose:

```text
[A, B]
   ↓
A finishes
   ↓
[B, C]
```

At this point:

```text
B → already has KV state and is decoding

C → brand-new request and still needs prefill
```

The scheduler therefore needs to handle requests that are at different stages of inference while the active set continuously changes.

---

## What We Will Build

### 1. Batched Decode

Instead of performing:

```text
model(A)
model(B)
```

we will decode multiple active requests together:

```text
token A ─┐
         ├── MODEL → next token A
token B ─┘          next token B
```

This lets one model execution advance multiple decoding sequences.

---

### 2. Dynamic Active Batches

The set of active requests will be allowed to change between engine iterations:

```text
[A, B]
   ↓
[B, C]
   ↓
[C, D]
```

Requests that finish leave immediately, while waiting requests can take their place.

The difficult part is preserving the correct inference state for requests that remain active.

---

### 3. Prefill and Decode Together

Not every active request requires the same amount of work.

A new request may need:

```text
C → PREFILL → entire prompt
```

while another request only needs:

```text
B → DECODE → one new token
```

The scheduler therefore needs to distinguish between:

```text
prefill work
```

and:

```text
decode work
```

rather than treating every request as identical.

---

### 4. Token-Budget-Aware Scheduling

So far we have mainly limited execution using:

```python
max_running = 2
```

This controls how many requests may be active.

But two requests can have completely different computational costs.

For example:

```text
Request A → decode 1 token

Request B → decode 1 token

Request C → prefill 14 prompt tokens
```

A more useful scheduling limit is therefore based on the number of tokens processed during an iteration.

We will introduce something like:

```python
max_tokens_per_step = 16
```

Then the scheduler can reason about work such as:

```text
A decode       = 1 token
B decode       = 1 token
C prefill      = 14 tokens
------------------------
total          = 16 tokens
```

This moves our scheduler from:

```text
How many requests can run?
```

toward:

```text
How much inference work can run during this iteration?
```

---

## What We Are Not Building Yet

If a new prompt contains more tokens than the remaining token budget, we will eventually need a way to split that prefill across multiple iterations.

That is **chunked prefill**.

We will not implement it here.

For Part 5B, a prefill must fit within the available scheduling budget. Chunked prefill will be introduced later as its own serving optimization.

We are also not yet implementing:

```text
paged KV-cache allocation
PagedAttention
prefix caching
KV-cache offloading
```

Those belong to later parts of the inference engine.

---

## Goal

By the end of Part 5B, our engine should conceptually operate like:

```text
new requests
      ↓
   waiting
      ↓
   scheduler
      ↓
token budget
      ↓
choose prefill + decode work
      ↓
batched model execution
      ↓
update request / KV state
      ↓
remove finished requests
      ↓
admit new requests
      ↓
repeat
```

The active batch will no longer be fixed.

Instead, the engine will repeatedly decide what work should execute during the next inference iteration.

In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DynamicCache,
)

In [2]:
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
).to("cuda")

model.eval()

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), ep

In [3]:
#configuring batching

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

In [4]:
print("Device:", model.device)
print("Dtype:", model.dtype)

print("EOS token:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token_id)

Device: cuda:0
Dtype: torch.float16
EOS token: 2
PAD token: 2


In [6]:
#this might look different from our prev notebooks as now we will actually follow the instruct model conventions

def create_request(request_id, prompt, max_new_tokens):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    return {
        "id": request_id,
        "prompt": prompt,

        "input_ids": inputs["input_ids"].to(model.device),
        "attention_mask": inputs["attention_mask"].to(model.device),

        "generated_tokens": [],
        "max_new_tokens": max_new_tokens,

        "finished": False,
    }

In [7]:
waiting = []
running = []
finished = []

max_running = 2

In [8]:
waiting.append(
    create_request(
        "A",
        "What is the capital of France? Answer briefly.",
        max_new_tokens=8,
    )
)

waiting.append(
    create_request(
        "B",
        "What is the largest planet in our solar system? Answer briefly.",
        max_new_tokens=8,
    )
)

waiting.append(
    create_request(
        "C",
        "What is machine learning? Answer in one sentence.",
        max_new_tokens=12,
    )
)

In [9]:
def schedule():

    while waiting and len(running) < max_running:

        request = waiting.pop(0)

        running.append(request)

In [10]:
schedule()

print("Running:", [r["id"] for r in running])
print("Waiting:", [r["id"] for r in waiting])

Running: ['A', 'B']
Waiting: ['C']


In [11]:
# building the prefill batch

batch = tokenizer.pad(
    {
        "input_ids": [
            request["input_ids"].squeeze(0)
            for request in running
        ],

        "attention_mask": [
            request["attention_mask"].squeeze(0)
            for request in running
        ],
    },
    padding=True,
    return_tensors="pt",
).to(model.device)

In [12]:
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)

input_ids: torch.Size([2, 43])
attention_mask: torch.Size([2, 43])


In [15]:
batch_cache = DynamicCache(
    config=model.config
)

with torch.inference_mode():

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        past_key_values=batch_cache,
        use_cache=True,
    )

# Save the KV cache created during prefill
batch_cache = outputs.past_key_values

In [16]:
#Getting the first generated token for both requests
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

print(next_tokens)
print(next_tokens.shape)


tensor([[504],
        [504]], device='cuda:0')
torch.Size([2, 1])


In [17]:
for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )

In [18]:
#extending attention mask for decode

batch_attention_mask = torch.cat(
    [
        batch["attention_mask"],

        torch.ones(
            (
                batch["attention_mask"].shape[0],
                1
            ),
            dtype=batch["attention_mask"].dtype,
            device=model.device,
        )
    ],
    dim=-1,
)

print(batch_attention_mask.shape)

torch.Size([2, 44])


In [19]:
#first batched decode

with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

In [20]:
batch_cache = outputs.past_key_values

In [21]:
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )

print(next_tokens)

tensor([[3575],
        [3995]], device='cuda:0')


In [23]:
#repeating batched decode

for _ in range(5):

    # Context grows by one token
    batch_attention_mask = torch.cat(
        [
            batch_attention_mask,

            torch.ones(
                (
                    batch_attention_mask.shape[0],
                    1
                ),
                dtype=batch_attention_mask.dtype,
                device=model.device,
            )
        ],
        dim=-1,
    )

    with torch.inference_mode():

        outputs = model(
            input_ids=next_tokens,
            attention_mask=batch_attention_mask,
            past_key_values=batch_cache,
            use_cache=True,
        )

    # Update KV cache
    batch_cache = outputs.past_key_values

    # Generate one token for every request
    next_tokens = outputs.logits[:, -1, :].argmax(
        dim=-1,
        keepdim=True,
    )

    # Save each batch row back to its request
    for i, request in enumerate(running):

        request["generated_tokens"].append(
            next_tokens[i].item()
        )

In [24]:
for request in running:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(f"\nRequest {request['id']}:")
    print(text)


Request A:
The capital of France is Paris.

Request B:
The largest planet in our solar system


## Batched Decode With KV Cache

In Part 5A, we managed to batch multiple prompts into one model forward pass.

However, that only proved that **prefill could be batched**.

For real continuous batching, we also need multiple active requests to remain batched during decoding.

Instead of:

```text
A → model()
B → model()
```

for every generated token, we want:

```text
A token ─┐
         ├── MODEL
B token ─┘
            ↓
      next token for A
      next token for B
```

The important idea is that after prefill, the previous context is already stored inside the KV cache.

So during decode we only need to send the **latest generated token for each sequence**.

---

## Initial Scheduler State

Our scheduler currently contains:

```text
Running: ['A', 'B']
Waiting: ['C']
```

So:

```text
A and B → active
C       → waiting
```

For this section, we deliberately keep the active batch fixed as:

```text
[A, B]
```

Before handling dynamic batches like:

```text
[A, B] → [B, C]
```

we first need to prove that A and B can repeatedly decode together.

---

## Building the Prefill Batch

The requests were tokenized using the instruct model's chat template and then padded into one batch.

We can inspect the result:

```python
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
```

Output:

```text
input_ids: torch.Size([2, 43])
attention_mask: torch.Size([2, 43])
```

The shape:

```text
[2, 43]
```

means:

```text
2  → two requests: A and B

43 → padded sequence length
```

The prompts are longer than the simple raw prompts we used in Part 5A because the chat template adds the formatting expected by the instruct model.

Conceptually, instead of only tokenizing:

```text
What is the capital of France?
```

the tokenizer creates something closer to:

```text
<user>
What is the capital of France?
<assistant>
```

with the exact formatting determined by the model's chat template.

---

## Batched Prefill

Create one KV cache for the batch:

```python
batch_cache = DynamicCache(
    config=model.config
)
```

Then perform one forward pass:

```python
with torch.inference_mode():

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        past_key_values=batch_cache,
        use_cache=True,
    )

batch_cache = outputs.past_key_values
```

This processes:

```text
A's complete prompt
B's complete prompt
```

together:

```text
A prompt ─┐
          ├── MODEL
B prompt ─┘
             ↓
        batched KV cache
```

This is the **prefill stage**.

Quick reminder: prefill is when the model processes the existing prompt and builds the KV states that later decode iterations can reuse.

---

## Generating the First Token

From the prefill logits:

```python
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

print(next_tokens)
print(next_tokens.shape)
```

Our output was:

```text
tensor([
    [504],
    [504]
], device='cuda:0')

torch.Size([2, 1])
```

The shape:

```text
[2, 1]
```

means:

```text
row 0 → one generated token for A
row 1 → one generated token for B
```

Even though both happened to generate token ID `504` during this step, they still belong to different sequences.

We store them back into their requests:

```python
for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )
```

The ordering matters:

```text
running[0] ↔ next_tokens[0] ↔ A

running[1] ↔ next_tokens[1] ↔ B
```

This mapping between scheduler requests and batch rows must remain correct throughout generation.

---

## Preparing the Attention Mask for Decode

During prefill our attention mask had shape:

```text
[2, 43]
```

After generating one new token, each sequence now has one additional visible position.

So we extend the mask:

```python
batch_attention_mask = torch.cat(
    [
        batch["attention_mask"],

        torch.ones(
            (
                batch["attention_mask"].shape[0],
                1
            ),
            dtype=batch["attention_mask"].dtype,
            device=model.device,
        )
    ],
    dim=-1,
)

print(batch_attention_mask.shape)
```

Output:

```text
torch.Size([2, 44])
```

So:

```text
before → [2, 43]

after  → [2, 44]
```

### Why does the mask grow if we only send one token?

Because during cached decoding:

```text
input_ids
```

contains only the token being processed **right now**.

But:

```text
attention_mask
```

describes the full context visible to the model:

```text
cached previous tokens
+
current token
```

The prompt is no longer sent again because its information is already represented inside the KV cache.

---

## First Batched Decode

Now comes the important step.

Instead of feeding both complete prompts again, we send:

```python
next_tokens
```

which has shape:

```text
[2, 1]
```

So the model receives:

```text
A → one new token
B → one new token
```

while their previous context comes from:

```python
batch_cache
```

The decode forward pass is:

```python
with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )
```

Then update the cache:

```python
batch_cache = outputs.past_key_values
```

Conceptually:

```text
            existing batched KV cache
                      +
A newest token ─┐
                ├── MODEL
B newest token ─┘
                      ↓
              updated KV cache
```

This is now **batched cached decoding**.

---

## Generate the Next Tokens

Again:

```python
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )

print(next_tokens)
```

Our result was:

```text
tensor([
    [3575],
    [3995]
], device='cuda:0')
```

This time A and B generated different token IDs:

```text
A → 3575

B → 3995
```

Both still came from **one model forward pass**.

---

## Repeating Batched Decode

Once this works, decoding can continue using the same pattern:

```python
for _ in range(5):

    # Every sequence receives one new token,
    # so its visible context becomes one position longer.
    batch_attention_mask = torch.cat(
        [
            batch_attention_mask,

            torch.ones(
                (
                    batch_attention_mask.shape[0],
                    1
                ),
                dtype=batch_attention_mask.dtype,
                device=model.device,
            )
        ],
        dim=-1,
    )

    with torch.inference_mode():

        outputs = model(
            input_ids=next_tokens,
            attention_mask=batch_attention_mask,
            past_key_values=batch_cache,
            use_cache=True,
        )

    # Save updated KV state
    batch_cache = outputs.past_key_values

    # Generate one token for every sequence
    next_tokens = outputs.logits[:, -1, :].argmax(
        dim=-1,
        keepdim=True,
    )

    # Map outputs back to their requests
    for i, request in enumerate(running):

        request["generated_tokens"].append(
            next_tokens[i].item()
        )
```

The important pattern is:

```text
                    PREFILL

A full prompt ─┐
               ├── MODEL
B full prompt ─┘
                    ↓
               KV CACHE
                    ↓
              [A1, B1]


                    DECODE

A1 ─┐
    ├── MODEL + KV CACHE
B1 ─┘
          ↓
       [A2, B2]


                    DECODE

A2 ─┐
    ├── MODEL + UPDATED KV CACHE
B2 ─┘
          ↓
       [A3, B3]
```

We are no longer running the two sequences independently.

Each decode iteration performs **one batched model call**.

---

## Decoding the Generated Outputs

Finally:

```python
for request in running:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(f"\nRequest {request['id']}:")
    print(text)
```

Our output was:

```text
Request A:
The capital of France is Paris.
```

and:

```text
Request B:
The largest planet in our solar system
```

The second answer is incomplete only because we deliberately performed a small fixed number of decode iterations.

Generation quality is not the goal of this section.

The important result is that both sequences were advanced together using the same batched decode operation.

---

## What We Have Now

We previously had only:

```text
batched prefill ✅
```

We now have:

```text
batched prefill           ✅

batched KV cache          ✅

batched decode            ✅

one token per active
request per iteration     ✅

one model forward for
multiple decode requests  ✅
```

So for a fixed active set:

```text
[A, B]
```

our execution now genuinely looks like:

```text
[A, B] → PREFILL

[A, B] → DECODE

[A, B] → DECODE

[A, B] → DECODE
```

instead of:

```text
A → model()
B → model()

A → model()
B → model()
```

---

## What Is Still Missing?

Our active batch has remained fixed:

```text
[A, B]
[A, B]
[A, B]
```

But continuous batching ultimately requires:

```text
[A, B]
[A, B]

A finishes

[B, C]
[B, C]
```

That introduces a harder problem.

`B` already has cached KV state from several decode iterations.

`C` is completely new and still needs prefill.

So next we need to understand how the engine handles:

```text
B → decode work

C → prefill work
```

while the active batch changes dynamically.

That is the next step toward a real dynamic continuous-batching loop.

In [25]:
# Make the next iteration finish A,
# while B still has several tokens left.

running[0]["max_new_tokens"] = (
    len(running[0]["generated_tokens"]) + 1
)

running[1]["max_new_tokens"] = (
    len(running[1]["generated_tokens"]) + 4
)

print(
    "A limit:",
    running[0]["max_new_tokens"]
)

print(
    "B limit:",
    running[1]["max_new_tokens"]
)

A limit: 8
B limit: 11


In [26]:
batch_attention_mask = torch.cat(
    [
        batch_attention_mask,

        torch.ones(
            (
                batch_attention_mask.shape[0],
                1
            ),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        )
    ],
    dim=-1,
)

In [27]:
with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

batch_cache = outputs.past_key_values

In [28]:
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )

In [29]:
keep_indices = []

for i, request in enumerate(running):

    reached_max_tokens = (
        len(request["generated_tokens"])
        >= request["max_new_tokens"]
    )

    generated_eos = (
        request["generated_tokens"][-1]
        == tokenizer.eos_token_id
    )

    if reached_max_tokens or generated_eos:

        request["finished"] = True
        finished.append(request)

    else:

        keep_indices.append(i)

In [30]:
print(keep_indices)

[1]


In [31]:
#removing finished rows from kv cache

keep_indices_tensor = torch.tensor(
    keep_indices,
    dtype=torch.long,
    device=model.device,
)

In [32]:
#pruning the one which was evicted
batch_cache.batch_select_indices(
    keep_indices_tensor
)

In [33]:
next_tokens = next_tokens[
    keep_indices_tensor
]

batch_attention_mask = batch_attention_mask[
    keep_indices_tensor
]

In [34]:
running = [
    request
    for request in running
    if not request["finished"]
]


print(
    "Running:",
    [request["id"] for request in running]
)

print(
    "Finished:",
    [request["id"] for request in finished]
)

print(
    "next_tokens:",
    next_tokens.shape
)

print(
    "attention_mask:",
    batch_attention_mask.shape
)

print(
    "cache batch size:",
    batch_cache.batch_size
)

Running: ['B']
Finished: ['A']
next_tokens: torch.Size([1, 1])
attention_mask: torch.Size([1, 50])
cache batch size: -1


In [35]:
print(
    "Actual KV batch dimension:",
    batch_cache.layers[0].keys.shape[0]
)

Actual KV batch dimension: 1


## Removing Finished Requests From a Batched KV Cache

We can now perform repeated batched decode for a fixed active set:

```text
[A, B]
[A, B]
[A, B]
...
```

But continuous batching requires requests to leave the active batch as soon as they finish.

Suppose:

```text
running = [A, B]
```

and A finishes before B.

The scheduler should transition to:

```text
running = [B]
```

However, removing A from the Python `running` list is **not enough**.

Our other batched state still contains one row for A and one row for B:

```text
running
row 0 → A
row 1 → B

next_tokens
row 0 → A
row 1 → B

attention_mask
row 0 → A
row 1 → B

KV cache
row 0 → A
row 1 → B
```

If A leaves the active batch, all of these structures must change together.

The fundamental rule is:

> Batch row `i` must always refer to the same request across the scheduler state, token tensors, attention mask and KV cache.

---

## Making A Finish Before B

For this experiment, we deliberately configure A to finish during the next decode iteration while B is allowed to continue.

```python
running[0]["max_new_tokens"] = (
    len(running[0]["generated_tokens"]) + 1
)

running[1]["max_new_tokens"] = (
    len(running[1]["generated_tokens"]) + 4
)

print(
    "A limit:",
    running[0]["max_new_tokens"]
)

print(
    "B limit:",
    running[1]["max_new_tokens"]
)
```

This is only for making the batch-removal behavior easy to observe.

It is not a scheduling policy.

---

## Run One More Batched Decode Step

Every active request is about to process one additional token, so first extend the attention mask:

```python
batch_attention_mask = torch.cat(
    [
        batch_attention_mask,

        torch.ones(
            (
                batch_attention_mask.shape[0],
                1
            ),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        )
    ],
    dim=-1,
)
```

Then perform another batched decode:

```python
with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

batch_cache = outputs.past_key_values
```

Generate the next token for each active request:

```python
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)
```

And map those outputs back to their requests:

```python
for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )
```

At this point the batched structures still contain:

```text
[A, B]
```

but A has now reached its generation limit.

---

## Determine Which Batch Rows Survive

We now inspect each request:

```python
keep_indices = []

for i, request in enumerate(running):

    reached_max_tokens = (
        len(request["generated_tokens"])
        >= request["max_new_tokens"]
    )

    generated_eos = (
        request["generated_tokens"][-1]
        == tokenizer.eos_token_id
    )

    if reached_max_tokens or generated_eos:

        request["finished"] = True
        finished.append(request)

    else:

        keep_indices.append(i)
```

Suppose our batch was:

```text
row 0 → A
row 1 → B
```

and A finished.

Then:

```python
print(keep_indices)
```

gives:

```text
[1]
```

This means:

> Keep only batch row 1.

That row currently belongs to B.

---

## Why We Need Batch Indices

We cannot simply say:

```text
remove A
```

to a tensor.

The GPU tensors understand positions such as:

```text
row 0
row 1
row 2
...
```

So the scheduler needs to map request state back to batch positions.

Here:

```text
A → batch index 0
B → batch index 1
```

After A finishes:

```text
keep index 1
```

---

## Convert the Indices Into a Tensor

```python
keep_indices_tensor = torch.tensor(
    keep_indices,
    dtype=torch.long,
    device=model.device,
)
```

This gives us the indices in a form the cache and PyTorch tensors can use.

---

## Remove A From the KV Cache

Now:

```python
batch_cache.batch_select_indices(
    keep_indices_tensor
)
```

`DynamicCache` stores key/value tensors whose first dimension is the batch dimension.

Conceptually:

```text
before:

KV cache
├── row 0 → A
└── row 1 → B
```

Selecting:

```text
[1]
```

produces:

```text
after:

KV cache
└── row 0 → B
```

Notice something subtle:

B used to be:

```text
row 1
```

but after removing A, B becomes:

```text
row 0
```

That is fine.

What matters is that **every other batched structure must now make the exact same change**.

---

## Remove A From the Other Batched State

First prune `next_tokens`:

```python
next_tokens = next_tokens[
    keep_indices_tensor
]
```

Before:

```text
[A token]
[B token]
```

After:

```text
[B token]
```

Then prune the attention mask:

```python
batch_attention_mask = batch_attention_mask[
    keep_indices_tensor
]
```

Before:

```text
A attention row
B attention row
```

After:

```text
B attention row
```

Finally update the scheduler's `running` list:

```python
running = [
    request
    for request in running
    if not request["finished"]
]
```

---

## Inspect the New State

```python
print(
    "Running:",
    [request["id"] for request in running]
)

print(
    "Finished:",
    [request["id"] for request in finished]
)

print(
    "next_tokens:",
    next_tokens.shape
)

print(
    "attention_mask:",
    batch_attention_mask.shape
)
```

Our result was:

```text
Running: ['B']

Finished: ['A']

next_tokens: torch.Size([1, 1])

attention_mask: torch.Size([1, 50])
```

This is exactly what we wanted.

The active batch originally contained:

```text
[A, B]
```

and now contains:

```text
[B]
```

Every structure has also changed its batch dimension from:

```text
2
```

to:

```text
1
```

---

## Inspecting the Actual KV Tensor

Calling:

```python
batch_cache.batch_size
```

returned:

```text
-1
```

in our environment.

That does **not** mean the cache contains `-1` requests or that the pruning failed.

The Transformers cache API's generic `batch_size` property can return `-1` when a cache layer does not expose a directly queryable batch-size attribute.

For our `DynamicCache`, the actual cached K/V tensor is the more useful thing to inspect.

```python
print(
    "KV tensor shape:",
    batch_cache.layers[0].keys.shape
)

print(
    "Actual KV batch dimension:",
    batch_cache.layers[0].keys.shape[0]
)
```

The key tensor is conceptually shaped:

```text
[
    batch_size,
    num_kv_heads,
    cached_sequence_length,
    head_dim
]
```

After removing A, the first dimension should therefore be:

```text
1
```

because only B remains.

---

## What Just Happened?

Before A finished:

```text
running
[A, B]

next_tokens
[A]
[B]

attention_mask
[A row]
[B row]

KV cache
[A row]
[B row]
```

Then:

```text
A finishes
```

We determined:

```text
keep_indices = [1]
```

and applied the same selection everywhere.

Afterward:

```text
running
[B]

next_tokens
[B]

attention_mask
[B row]

KV cache
[B row]
```

The batch has been **compacted**.

---

## Why This Matters

This is the first time our batch actually changed while preserving the state of an already-running request.

We have moved from:

```text
[A, B]
```

to:

```text
[B]
```

without recomputing B's prompt or throwing away B's KV cache.

That is important because continuous batching depends on requests independently entering and leaving the active set.

However, we have only solved the easier half:

```text
[A, B] → [B]
```

The next problem is harder.

C is still waiting:

```text
running = [B]
waiting = [C]
```

B is already decoding and has a populated KV cache.

C has never executed and therefore still needs **prefill**.

So next we need to achieve:

```text
[B] → [B, C]
```

while handling:

```text
B → decode

C → prefill
```

inside the same serving system.

That is where dynamic continuous batching starts combining scheduling decisions with different kinds of model work.

In [36]:
# B is already running.
# Admit C into the active set.

c_request = waiting.pop(0)
running.append(c_request)

b_request = running[0]
c_request = running[1]

print("Running:", [r["id"] for r in running])
print("Waiting:", [r["id"] for r in waiting])

Running: ['B', 'C']
Waiting: []


In [37]:
b_attention_mask = torch.cat(
    [
        batch_attention_mask,
        torch.ones(
            (1, 1),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        ),
    ],
    dim=-1,
)

In [38]:
with torch.inference_mode():

    b_outputs = model(
        input_ids=next_tokens,
        attention_mask=b_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

b_cache = b_outputs.past_key_values

b_next_token = b_outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

b_request["generated_tokens"].append(
    b_next_token.item()
)

In [39]:
print("B cache length:", b_cache.get_seq_length())
print("B next token shape:", b_next_token.shape)

B cache length: 51
B next token shape: torch.Size([1, 1])


In [40]:
#prefilling C to the same physical cache length

target_cache_length = b_cache.get_seq_length()

print("Target cache length:", target_cache_length)
print("C prompt length:", c_request["input_ids"].shape[-1])

Target cache length: 51
C prompt length: 40


In [41]:
c_prompt_length = c_request["input_ids"].shape[-1]

if c_prompt_length > target_cache_length:
    raise ValueError(
        "C's prompt is longer than the available dense-cache length. "
        "We will handle oversized prefills later with scheduling/chunking."
    )

In [42]:
#left padding with difference


import torch.nn.functional as F

pad_length = target_cache_length - c_prompt_length

c_input_ids = F.pad(
    c_request["input_ids"],
    (pad_length, 0),
    value=tokenizer.pad_token_id,
)

c_attention_mask = F.pad(
    c_request["attention_mask"],
    (pad_length, 0),
    value=0,
)

In [43]:
print("C input shape:", c_input_ids.shape)
print("C mask shape:", c_attention_mask.shape)

C input shape: torch.Size([1, 51])
C mask shape: torch.Size([1, 51])


In [44]:
#since seq lenght is same now we will prefill c

c_cache = DynamicCache(
    config=model.config
)

with torch.inference_mode():

    c_outputs = model(
        input_ids=c_input_ids,
        attention_mask=c_attention_mask,
        past_key_values=c_cache,
        use_cache=True,
    )

c_cache = c_outputs.past_key_values

c_next_token = c_outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

c_request["generated_tokens"].append(
    c_next_token.item()
)

print("B cache length:", b_cache.get_seq_length())
print("C cache length:", c_cache.get_seq_length())

B cache length: 51
C cache length: 51


In [45]:
#merge b and c into one batch

merged_cache_data = []

for b_layer, c_layer in zip(
    b_cache.layers,
    c_cache.layers,
):

    merged_keys = torch.cat(
        [
            b_layer.keys,
            c_layer.keys,
        ],
        dim=0,
    )

    merged_values = torch.cat(
        [
            b_layer.values,
            c_layer.values,
        ],
        dim=0,
    )

    merged_cache_data.append(
        (
            merged_keys,
            merged_values,
        )
    )

In [46]:
#constructing new cache

batch_cache = DynamicCache(
    ddp_cache_data=merged_cache_data,
    config=model.config,
)

In [47]:
next_tokens = torch.cat(
    [
        b_next_token,
        c_next_token,
    ],
    dim=0,
)

batch_attention_mask = torch.cat(
    [
        b_attention_mask,
        c_attention_mask,
    ],
    dim=0,
)


print(
    "Running:",
    [r["id"] for r in running]
)

print(
    "next_tokens:",
    next_tokens.shape
)

print(
    "attention_mask:",
    batch_attention_mask.shape
)

print(
    "KV batch dimension:",
    batch_cache.layers[0].keys.shape[0]
)

print(
    "KV sequence length:",
    batch_cache.get_seq_length()
)

Running: ['B', 'C']
next_tokens: torch.Size([2, 1])
attention_mask: torch.Size([2, 51])
KV batch dimension: 2
KV sequence length: 51


In [48]:
batch_attention_mask = torch.cat(
    [
        batch_attention_mask,

        torch.ones(
            (
                batch_attention_mask.shape[0],
                1,
            ),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        ),
    ],
    dim=-1,
)

In [49]:
#batched decode

with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

batch_cache = outputs.past_key_values

In [50]:
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )


print("Running:", [r["id"] for r in running])
print("next_tokens:", next_tokens.shape)
print("KV batch:", batch_cache.layers[0].keys.shape[0])

for request in running:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(f"\nRequest {request['id']}:")
    print(text)

Running: ['B', 'C']
next_tokens: torch.Size([2, 1])
KV batch: 2

Request B:
The largest planet in our solar system is Jupiter.

Request C:
Machine learning


## Dynamic Admission and Prefill/Decode Coexistence

So far, we have handled two important cases:

```text
[A, B] → batched decode
```

and:

```text
[A, B] → [B]
```

when A finishes.

Now we need to solve the other half of continuous batching:

```text
[B] → [B, C]
```

This looks simple from the scheduler's point of view.

We could just move C from:

```text
waiting
```

into:

```text
running
```

However, the model state makes this much harder.

At this point:

```text
B → already has KV state and is decoding

C → has never executed and still needs prefill
```

So even though both requests are now considered active, they require different kinds of model work.

---

## Starting State

After removing A, our engine currently looks like:

```text
running = [B]

waiting = [C]
```

B already has:

```text
next token
attention mask
KV cache
```

representing all of its previous inference work.

C only has its original tokenized prompt.

Our goal is to reach:

```text
running = [B, C]
```

without throwing away B's existing KV cache.

---

## Admit C Into the Active Set

First move C from `waiting` into `running`:

```python
c_request = waiting.pop(0)

running.append(c_request)

b_request = running[0]
c_request = running[1]

print(
    "Running:",
    [request["id"] for request in running]
)

print(
    "Waiting:",
    [request["id"] for request in waiting]
)
```

At this point:

```text
Running: ['B', 'C']
Waiting: []
```

But this does **not** mean they are ready for one model call yet.

Their inference states are different:

```text
B → DECODE

C → PREFILL
```

---

### Why B and C Cannot Immediately Share the Same Cache

B's cache may already represent something like:

```text
prompt
+
many generated tokens
```

while C has:

```text
no cache at all
```

A normal dense KV tensor needs consistent dimensions.

Conceptually:

```text
B cache:
[1, kv_heads, 50, head_dim]

C cache:
does not exist yet
```

We cannot simply concatenate those.

So we first advance each request according to the kind of work it needs.

---

### Decode B

B is already in the decode phase.

Quick reminder:

During decode, the complete previous context is already represented by the KV cache.

Therefore we only feed:

```text
the newest token
+
existing KV cache
```

instead of sending the full sequence again.

First extend B's attention mask by one position:

```python
b_attention_mask = torch.cat(
    [
        batch_attention_mask,

        torch.ones(
            (1, 1),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        ),
    ],
    dim=-1,
)
```

Then perform one decode step:

```python
with torch.inference_mode():

    b_outputs = model(
        input_ids=next_tokens,
        attention_mask=b_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

b_cache = b_outputs.past_key_values
```

Generate B's next token:

```python
b_next_token = b_outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)
```

Store it:

```python
b_request["generated_tokens"].append(
    b_next_token.item()
)
```

At this point B has advanced by one real decode iteration.

---

### C Still Needs Prefill

C has never been through the model.

Therefore its first execution must process its complete prompt.

This is the **prefill stage**.

However, before B and C can later be represented inside the same dense KV cache, their physical cache sequence lengths need to match.

Get B's current cache length:

```python
target_cache_length = b_cache.get_seq_length()

print(
    "Target cache length:",
    target_cache_length
)

print(
    "C prompt length:",
    c_request["input_ids"].shape[-1]
)
```

Suppose B's cache currently contains:

```text
51 positions
```

while C's prompt contains fewer tokens.

We can left-pad C to the same physical length.

---

## Why Are We Padding the Cache Length?

Remember that a dense KV tensor looks conceptually like:

```text
[
    batch,
    kv_heads,
    sequence_length,
    head_dim
]
```

If B has:

```text
sequence_length = 51
```

and C has:

```text
sequence_length = 40
```

we cannot concatenate them along the batch dimension because the other dimensions do not match.

So for this experiment we align C's physical sequence length to B's cache length.

The padding positions are ignored using the attention mask.

---

### Ensure C Fits

```python
c_prompt_length = (
    c_request["input_ids"].shape[-1]
)

if c_prompt_length > target_cache_length:

    raise ValueError(
        "C's prompt is longer than the available "
        "dense-cache length."
    )
```

For this section, C must fit inside the current dense sequence length.

Handling a prompt larger than the available scheduling budget will be covered in the next scheduler section.

---

### Left-Pad C

```python
import torch.nn.functional as F

pad_length = (
    target_cache_length
    - c_prompt_length
)
```

Pad its token IDs:

```python
c_input_ids = F.pad(
    c_request["input_ids"],
    (pad_length, 0),
    value=tokenizer.pad_token_id,
)
```

Pad its attention mask:

```python
c_attention_mask = F.pad(
    c_request["attention_mask"],
    (pad_length, 0),
    value=0,
)
```

The `0` values in the attention mask tell the model that those padded positions are not real context.

After this:

```text
B physical length = 51

C physical length = 51
```

even though C may contain fewer actual prompt tokens.

---

### Prefill C

Create a fresh cache for C:

```python
c_cache = DynamicCache(
    config=model.config
)
```

Then perform C's prefill:

```python
with torch.inference_mode():

    c_outputs = model(
        input_ids=c_input_ids,
        attention_mask=c_attention_mask,
        past_key_values=c_cache,
        use_cache=True,
    )

c_cache = c_outputs.past_key_values
```

Generate C's first token:

```python
c_next_token = (
    c_outputs.logits[:, -1, :]
    .argmax(
        dim=-1,
        keepdim=True,
    )
)
```

Save it:

```python
c_request["generated_tokens"].append(
    c_next_token.item()
)
```

Now:

```text
B → completed one decode step

C → completed its prefill
```

and both have KV caches.

---

### Merge the Two KV Caches

At this point their cache tensors are aligned:

```text
B cache:
[1, kv_heads, 51, head_dim]

C cache:
[1, kv_heads, 51, head_dim]
```

So we can concatenate them along:

```text
dimension 0
```

which is the batch dimension.

For every transformer layer:

```python
merged_cache_data = []

for b_layer, c_layer in zip(
    b_cache.layers,
    c_cache.layers,
):

    merged_keys = torch.cat(
        [
            b_layer.keys,
            c_layer.keys,
        ],
        dim=0,
    )

    merged_values = torch.cat(
        [
            b_layer.values,
            c_layer.values,
        ],
        dim=0,
    )

    merged_cache_data.append(
        (
            merged_keys,
            merged_values,
        )
    )
```

The result changes:

```text
B:
[1, ...]

C:
[1, ...]
```

into:

```text
[B, C]:
[2, ...]
```

---

### Rebuild the Batched Cache

Now construct a new `DynamicCache` from the merged K/V tensors:

```python
batch_cache = DynamicCache(
    ddp_cache_data=merged_cache_data,
    config=model.config,
)
```

The new batch mapping is:

```text
cache row 0 → B
cache row 1 → C
```

This must match:

```text
running[0] → B
running[1] → C
```

As before, maintaining this row-to-request mapping is critical.

---

### Merge the Pending Tokens

B has:

```python
b_next_token
```

and C has:

```python
c_next_token
```

Combine them:

```python
next_tokens = torch.cat(
    [
        b_next_token,
        c_next_token,
    ],
    dim=0,
)
```

Now:

```text
next_tokens.shape = [2, 1]
```

with:

```text
row 0 → B
row 1 → C
```

---

### Merge the Attention Masks

Similarly:

```python
batch_attention_mask = torch.cat(
    [
        b_attention_mask,
        c_attention_mask,
    ],
    dim=0,
)
```

Both requests now share one dense batched representation.

Check everything:

```python
print(
    "Running:",
    [request["id"] for request in running]
)

print(
    "next_tokens:",
    next_tokens.shape
)

print(
    "attention_mask:",
    batch_attention_mask.shape
)

print(
    "KV batch dimension:",
    batch_cache.layers[0].keys.shape[0]
)

print(
    "KV sequence length:",
    batch_cache.get_seq_length()
)
```

At this stage we should have:

```text
Running: ['B', 'C']

next_tokens:
[2, 1]

KV batch dimension:
2
```

So we have successfully converted:

```text
[B]
```

into:

```text
[B, C]
```

without recomputing B's prompt.

---

### Proving B and C Can Decode Together

Now extend the attention mask by one token:

```python
batch_attention_mask = torch.cat(
    [
        batch_attention_mask,

        torch.ones(
            (
                batch_attention_mask.shape[0],
                1,
            ),
            dtype=batch_attention_mask.dtype,
            device=model.device,
        ),
    ],
    dim=-1,
)
```

Then perform one batched decode:

```python
with torch.inference_mode():

    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        past_key_values=batch_cache,
        use_cache=True,
    )

batch_cache = outputs.past_key_values
```

Generate one next token for both requests:

```python
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)
```

Store them:

```python
for i, request in enumerate(running):

    request["generated_tokens"].append(
        next_tokens[i].item()
    )
```

Inspect:

```python
print(
    "Running:",
    [request["id"] for request in running]
)

print(
    "next_tokens:",
    next_tokens.shape
)

print(
    "KV batch:",
    batch_cache.layers[0].keys.shape[0]
)
```

Our result was:

```text
Running: ['B', 'C']

next_tokens:
torch.Size([2, 1])

KV batch:
2
```

So B and C are now genuinely part of the same batched decode.

---

### Generated Output

Decoding their current token histories:

```python
for request in running:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(
        f"\nRequest {request['id']}:"
    )

    print(text)
```

produced:

```text
Request B:
The largest planet in our solar system is Jupiter.
```

and:

```text
Request C:
Machine learning
```

C's answer is incomplete because we have only run a small number of decode iterations.

Again, the point here is not generation length.

The important result is the state transition.

---

### What We Have Achieved

We started from:

```text
running = [B]

waiting = [C]
```

where:

```text
B → decode
C → not started
```

Then during the admission step:

```text
B → decode one token

C → prefill its prompt
```

Their inference states were aligned and merged:

```text
B cache ─┐
         ├── batched KV cache
C cache ─┘
```

After that, both could participate in normal batched decode:

```text
B token ─┐
         ├── MODEL
C token ─┘
             ↓
      next token for B
      next token for C
```

So we have now successfully demonstrated the dynamic transition:

```text
[A, B]
   ↓
A finishes

[B]
   ↓
C admitted

[B, C]
```

while preserving B's already-computed inference state.

---

## One Important Limitation

The dense-cache merging we performed here works as a useful bridge for our Transformers implementation, but it is awkward.

Every time a request joins, we may need to manipulate and rebuild dense K/V tensors.

A production inference engine does not want to keep physically rebuilding giant contiguous KV tensors every time the active set changes.

That problem is one of the reasons inference engines use more sophisticated KV-cache allocation schemes.

We will address that later when we build:

```text
KV-cache management
      ↓
paged allocation
      ↓
PagedAttention-style block mapping
```

For now, the important continuous-batching concept is complete:

> Existing decode work and new prefill work can coexist in the scheduler, after which both requests can join the same active decode batch.

The final piece of Part 5B is to stop making these decisions manually and let the **scheduler decide them using a token budget**.

In [51]:
# ============================================================
# PART 5B — FINAL DYNAMIC CONTINUOUS BATCHING SCHEDULER
#
# Covers:
# - token-budget-aware scheduling
# - decode cost = 1 token / active request
# - prefill cost = prompt token count
# - prefills that do not fit remain waiting
# - dynamic admission
# - batched decode
# - request completion
# - KV-cache compaction
# - prefill + decode coexistence
# - final execution trace
#
# NOTE:
# No chunked prefill yet.
# A prompt must fit completely inside one step's remaining
# token budget.
# ============================================================

import torch
import torch.nn.functional as F
from transformers import DynamicCache


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

max_running = 2

# Maximum logical tokens the scheduler may process per step.
#
# Decode request = 1 token.
# Prefill request = number of prompt tokens.
max_tokens_per_step = 50


# ------------------------------------------------------------
# REQUEST CREATION
# ------------------------------------------------------------

def create_request(request_id, prompt, max_new_tokens):

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    return {
        "id": request_id,
        "prompt": prompt,

        "input_ids": input_ids,
        "attention_mask": attention_mask,

        # Actual number of prompt tokens.
        "prefill_tokens": int(
            attention_mask.sum().item()
        ),

        "generated_tokens": [],
        "max_new_tokens": max_new_tokens,

        "finished": False,
    }


# ------------------------------------------------------------
# REQUESTS
# ------------------------------------------------------------

waiting = [
    create_request(
        "A",
        "What is the capital of France? Answer briefly.",
        max_new_tokens=4,
    ),

    create_request(
        "B",
        "What is the largest planet in our solar system? Answer briefly.",
        max_new_tokens=7,
    ),

    create_request(
        "C",
        "What is machine learning? Answer briefly.",
        max_new_tokens=6,
    ),
]

running = []
finished = []


# ------------------------------------------------------------
# BATCH STATE
# ------------------------------------------------------------

# These represent the currently-running dense batch.

batch_cache = None

batch_attention_mask = None

# next_tokens contains one pending token for every running request.
#
# shape:
# [num_running_requests, 1]
next_tokens = None


# ------------------------------------------------------------
# POSITION IDS FOR LEFT-PADDED PREFILL
# ------------------------------------------------------------

def make_prefill_position_ids(attention_mask):

    position_ids = (
        attention_mask.long().cumsum(dim=-1) - 1
    )

    position_ids.masked_fill_(
        attention_mask == 0,
        0,
    )

    return position_ids


# ------------------------------------------------------------
# MERGE TWO DYNAMIC CACHES
# ------------------------------------------------------------

def merge_caches(existing_cache, new_cache):

    merged_cache_data = []

    for old_layer, new_layer in zip(
        existing_cache.layers,
        new_cache.layers,
    ):

        merged_keys = torch.cat(
            [
                old_layer.keys,
                new_layer.keys,
            ],
            dim=0,
        )

        merged_values = torch.cat(
            [
                old_layer.values,
                new_layer.values,
            ],
            dim=0,
        )

        merged_cache_data.append(
            (
                merged_keys,
                merged_values,
            )
        )

    return DynamicCache(
        ddp_cache_data=merged_cache_data,
        config=model.config,
    )


# ------------------------------------------------------------
# PREFILL NEW REQUESTS
# ------------------------------------------------------------

@torch.inference_mode()
def prefill_requests(requests, target_cache_length=None):

    # If there is no existing decode batch,
    # use the longest new prompt as our physical width.

    longest_prompt = max(
        request["input_ids"].shape[-1]
        for request in requests
    )

    if target_cache_length is None:

        target_length = longest_prompt

    else:

        # Our dense-cache bridge requires the new prompts
        # to fit inside the existing physical cache length.

        if longest_prompt > target_cache_length:

            raise RuntimeError(
                "New prefill is longer than the current dense "
                "KV-cache length. A real paged engine handles this "
                "without rebuilding dense cache tensors."
            )

        target_length = target_cache_length


    padded_input_ids = []

    padded_attention_masks = []


    for request in requests:

        current_length = (
            request["input_ids"].shape[-1]
        )

        pad_length = (
            target_length - current_length
        )

        ids = F.pad(
            request["input_ids"],
            (pad_length, 0),
            value=tokenizer.pad_token_id,
        )

        mask = F.pad(
            request["attention_mask"],
            (pad_length, 0),
            value=0,
        )

        padded_input_ids.append(ids)

        padded_attention_masks.append(mask)


    input_ids = torch.cat(
        padded_input_ids,
        dim=0,
    )

    attention_mask = torch.cat(
        padded_attention_masks,
        dim=0,
    )


    position_ids = make_prefill_position_ids(
        attention_mask
    )


    cache = DynamicCache(
        config=model.config
    )


    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=cache,
        use_cache=True,
    )


    cache = outputs.past_key_values


    generated = (
        outputs.logits[:, -1, :]
        .argmax(
            dim=-1,
            keepdim=True,
        )
    )


    # Map each batch row back to its request.

    for i, request in enumerate(requests):

        request["generated_tokens"].append(
            generated[i].item()
        )


    return (
        cache,
        attention_mask,
        generated,
    )


# ------------------------------------------------------------
# ONE BATCHED DECODE ITERATION
# ------------------------------------------------------------

@torch.inference_mode()
def decode_running_batch(
    running,
    batch_cache,
    batch_attention_mask,
    next_tokens,
):

    batch_size = len(running)


    # Every running sequence is processing exactly
    # one new token during decode.

    batch_attention_mask = torch.cat(
        [
            batch_attention_mask,

            torch.ones(
                (
                    batch_size,
                    1,
                ),
                dtype=batch_attention_mask.dtype,
                device=model.device,
            ),
        ],
        dim=-1,
    )


    # Logical position of the token currently being decoded.
    #
    # This matters because different rows may contain
    # different amounts of left padding.

    position_ids = (
        batch_attention_mask.long()
        .sum(
            dim=-1,
            keepdim=True,
        )
        - 1
    )


    outputs = model(
        input_ids=next_tokens,
        attention_mask=batch_attention_mask,
        position_ids=position_ids,
        past_key_values=batch_cache,
        use_cache=True,
    )


    batch_cache = outputs.past_key_values


    next_tokens = (
        outputs.logits[:, -1, :]
        .argmax(
            dim=-1,
            keepdim=True,
        )
    )


    for i, request in enumerate(running):

        request["generated_tokens"].append(
            next_tokens[i].item()
        )


    return (
        batch_cache,
        batch_attention_mask,
        next_tokens,
    )


# ------------------------------------------------------------
# REMOVE FINISHED REQUESTS + COMPACT BATCH
# ------------------------------------------------------------

def compact_finished(
    running,
    finished,
    batch_cache,
    batch_attention_mask,
    next_tokens,
):

    if not running:

        return (
            running,
            finished,
            None,
            None,
            None,
            [],
        )


    keep_indices = []

    finished_this_step = []


    for i, request in enumerate(running):

        reached_max_tokens = (
            len(request["generated_tokens"])
            >= request["max_new_tokens"]
        )

        generated_eos = (
            len(request["generated_tokens"]) > 0
            and
            request["generated_tokens"][-1]
            == tokenizer.eos_token_id
        )


        if reached_max_tokens or generated_eos:

            request["finished"] = True

            finished.append(request)

            finished_this_step.append(
                request["id"]
            )

        else:

            keep_indices.append(i)


    # Nobody finished.

    if len(keep_indices) == len(running):

        return (
            running,
            finished,
            batch_cache,
            batch_attention_mask,
            next_tokens,
            finished_this_step,
        )


    # Everybody finished.

    if len(keep_indices) == 0:

        return (
            [],
            finished,
            None,
            None,
            None,
            finished_this_step,
        )


    # Some requests survived.
    #
    # Keep exactly the corresponding rows in every
    # batch-aligned structure.

    index_tensor = torch.tensor(
        keep_indices,
        dtype=torch.long,
        device=model.device,
    )


    batch_cache.batch_select_indices(
        index_tensor
    )


    batch_attention_mask = (
        batch_attention_mask[index_tensor]
    )


    next_tokens = (
        next_tokens[index_tensor]
    )


    running = [
        running[i]
        for i in keep_indices
    ]


    return (
        running,
        finished,
        batch_cache,
        batch_attention_mask,
        next_tokens,
        finished_this_step,
    )


# ------------------------------------------------------------
# CHECK THAT PREFILLS ARE SCHEDULABLE AT ALL
# ------------------------------------------------------------

for request in waiting:

    if request["prefill_tokens"] > max_tokens_per_step:

        raise ValueError(
            f"Request {request['id']} has "
            f"{request['prefill_tokens']} prompt tokens, "
            f"which exceeds max_tokens_per_step="
            f"{max_tokens_per_step}. "
            f"This would require chunked prefill."
        )


# ------------------------------------------------------------
# FINAL CONTINUOUS-BATCHING ENGINE LOOP
# ------------------------------------------------------------

step = 0

max_steps = 100


while waiting or running:

    step += 1

    if step > max_steps:

        raise RuntimeError(
            "Engine exceeded max_steps."
        )


    # Fresh token budget for this engine iteration.

    remaining_budget = (
        max_tokens_per_step
    )


    decoded_ids = []

    prefilled_ids = []

    deferred_message = None

    finished_ids = []

    made_progress = False


    # ========================================================
    # 1. DECODE CURRENT RUNNING REQUESTS
    # ========================================================

    if running:

        decode_cost = len(running)

        if decode_cost > remaining_budget:

            raise RuntimeError(
                "Token budget is smaller than the active "
                "decode batch."
            )


        decoded_ids = [
            request["id"]
            for request in running
        ]


        (
            batch_cache,
            batch_attention_mask,
            next_tokens,
        ) = decode_running_batch(
            running,
            batch_cache,
            batch_attention_mask,
            next_tokens,
        )


        remaining_budget -= decode_cost

        made_progress = True


        # Finished requests leave immediately.

        (
            running,
            finished,
            batch_cache,
            batch_attention_mask,
            next_tokens,
            newly_finished,
        ) = compact_finished(
            running,
            finished,
            batch_cache,
            batch_attention_mask,
            next_tokens,
        )


        finished_ids.extend(
            newly_finished
        )


    # ========================================================
    # 2. TOKEN-BUDGET-AWARE PREFILL ADMISSION
    # ========================================================

    requests_to_prefill = []


    while (
        waiting
        and
        len(running) + len(requests_to_prefill)
        < max_running
    ):

        request = waiting[0]

        prefill_cost = (
            request["prefill_tokens"]
        )


        # No chunked prefill in Part 5B.
        #
        # If the whole prompt does not fit in the remaining
        # budget, it simply stays in WAITING.

        if prefill_cost > remaining_budget:

            deferred_message = (
                f"{request['id']} prefill deferred "
                f"({prefill_cost} tokens > "
                f"{remaining_budget} remaining)"
            )

            break


        # Dense Hugging Face cache limitation:
        #
        # when joining an existing dense decode batch,
        # the new prompt must currently fit inside the
        # existing physical cache width.
        #
        # Production paged KV-cache systems do not need
        # this exact restriction.

        if (
            running
            and batch_cache is not None
            and request["input_ids"].shape[-1]
            > batch_cache.get_seq_length()
        ):

            deferred_message = (
                f"{request['id']} waiting for dense-cache "
                f"alignment"
            )

            break


        waiting.pop(0)

        requests_to_prefill.append(
            request
        )

        remaining_budget -= prefill_cost


    # ========================================================
    # 3. EXECUTE THE NEW PREFILLS
    # ========================================================

    if requests_to_prefill:

        prefilled_ids = [
            request["id"]
            for request in requests_to_prefill
        ]


        # Existing running requests already have a cache.
        #
        # New prefills are physically padded to that cache
        # length before merging.

        if running and batch_cache is not None:

            target_cache_length = (
                batch_cache.get_seq_length()
            )

        else:

            target_cache_length = None


        (
            new_cache,
            new_attention_mask,
            new_next_tokens,
        ) = prefill_requests(
            requests_to_prefill,
            target_cache_length,
        )


        # ----------------------------------------------------
        # Existing decode batch + newly-prefilled requests
        # ----------------------------------------------------

        if running:

            batch_cache = merge_caches(
                batch_cache,
                new_cache,
            )


            batch_attention_mask = torch.cat(
                [
                    batch_attention_mask,
                    new_attention_mask,
                ],
                dim=0,
            )


            next_tokens = torch.cat(
                [
                    next_tokens,
                    new_next_tokens,
                ],
                dim=0,
            )


            running.extend(
                requests_to_prefill
            )


        # ----------------------------------------------------
        # No existing batch: new prefills become the batch
        # ----------------------------------------------------

        else:

            batch_cache = new_cache

            batch_attention_mask = (
                new_attention_mask
            )

            next_tokens = new_next_tokens

            running = requests_to_prefill


        made_progress = True


        # A request can theoretically produce EOS during
        # its initial prefill prediction, so check again.

        (
            running,
            finished,
            batch_cache,
            batch_attention_mask,
            next_tokens,
            newly_finished,
        ) = compact_finished(
            running,
            finished,
            batch_cache,
            batch_attention_mask,
            next_tokens,
        )


        finished_ids.extend(
            newly_finished
        )


    # ========================================================
    # 4. TRACE THE SCHEDULER DECISION
    # ========================================================

    print(
        f"\nSTEP {step}"
    )

    print(
        "  decode :",
        decoded_ids
    )

    print(
        "  prefill:",
        prefilled_ids
    )

    print(
        "  finished:",
        finished_ids
    )

    print(
        "  running:",
        [
            request["id"]
            for request in running
        ]
    )

    print(
        "  waiting:",
        [
            request["id"]
            for request in waiting
        ]
    )

    print(
        "  budget used:",
        max_tokens_per_step
        - remaining_budget,
        "/",
        max_tokens_per_step,
    )


    if deferred_message:

        print(
            "  scheduler:",
            deferred_message
        )


    if batch_cache is not None:

        print(
            "  KV batch:",
            batch_cache.layers[0].keys.shape[0],
            "| KV length:",
            batch_cache.get_seq_length(),
        )


    # Protect ourselves from a scheduler deadlock.

    if not made_progress:

        raise RuntimeError(
            "Scheduler made no progress. "
            "A waiting request cannot currently be scheduled."
        )


# ------------------------------------------------------------
# FINAL RESULTS
# ------------------------------------------------------------

print(
    "\n\n========== FINISHED =========="
)

for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(
        f"\nRequest {request['id']}:"
    )

    print(text)


STEP 1
  decode : []
  prefill: ['A']
  finished: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 40 / 50
  scheduler: B prefill deferred (43 tokens > 10 remaining)
  KV batch: 1 | KV length: 40

STEP 2
  decode : ['A']
  prefill: []
  finished: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 1 / 50
  scheduler: B waiting for dense-cache alignment
  KV batch: 1 | KV length: 41

STEP 3
  decode : ['A']
  prefill: []
  finished: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 1 / 50
  scheduler: B waiting for dense-cache alignment
  KV batch: 1 | KV length: 42

STEP 4
  decode : ['A']
  prefill: ['B']
  finished: ['A']
  running: ['B']
  waiting: ['C']
  budget used: 44 / 50
  scheduler: C prefill deferred (38 tokens > 6 remaining)
  KV batch: 1 | KV length: 43

STEP 5
  decode : ['B']
  prefill: ['C']
  finished: []
  running: ['B', 'C']
  waiting: []
  budget used: 39 / 50
  KV batch: 2 | KV length: 44

STEP 6
  decode : ['B', 'C']
  prefill: []
  finished: 

## Token-Budget-Aware Dynamic Continuous Batching

We now have almost all of the individual mechanisms required for continuous batching.

So far, we have built:

```text
batched prefill
      ↓
batched decode
      ↓
request completion
      ↓
KV-cache compaction
      ↓
dynamic request admission
```

However, our scheduling decisions have mostly been made manually.

The final step is to let the scheduler decide how much work should execute during each engine iteration.

---

### Why `max_running` Is Not Enough

Earlier, our scheduler used:

```python
max_running = 2
```

This limits the number of active requests.

But the number of requests does not tell us how much actual model work they require.

Consider:

```text
Request A → decode
cost = 1 token

Request B → decode
cost = 1 token

Request C → prefill
cost = 38 tokens
```

All three are requests, but their computational work is very different.

A decode request usually processes only the newest token because its previous context is already stored in the KV cache.

A new request, however, must process its prompt during prefill.

Therefore we introduce another scheduler constraint:

```python
max_tokens_per_step = 50
```

This represents the maximum amount of token work our scheduler will allow during one engine iteration.

The scheduler can now reason in terms of:

```text
How many tokens am I processing this step?
```

rather than only:

```text
How many requests are running?
```

---

### Scheduling Cost

For our current engine, we use:

```text
DECODE

1 active request
=
1 scheduled token
```

because each decoding request advances by one token per iteration.

For prefill:

```text
PREFILL

cost
=
number of prompt tokens
```

So a scheduler iteration could look like:

```text
token budget = 50

B decode      = 1
C prefill     = 38
-----------------
used          = 39

remaining     = 11
```

This is exactly what happened during our execution.

---

## Prefill Deferral

There is one important rule in Part 5B.

A new prefill must fit completely inside the remaining token budget.

Suppose:

```text
remaining budget = 10

B prompt = 43 tokens
```

We do **not** process only ten tokens of B.

Instead:

```text
B remains waiting
```

until an iteration has enough budget for the complete prefill.

Splitting a large prefill across multiple iterations is called **chunked prefill**.

That is a later part of the engine and is intentionally not implemented here.

---

## Final Dynamic Continuous-Batching Engine

Our serving loop now conceptually performs:

```text
START ITERATION
      │
      ▼
new token budget
      │
      ▼
decode existing requests
      │
      ▼
subtract decode cost
      │
      ▼
check finished requests
      │
      ▼
compact active batch
      │
      ▼
inspect waiting requests
      │
      ▼
does prefill fit budget?
   /             \
 yes              no
  │                │
prefill         keep waiting
  │
  ▼
admit request
  │
  ▼
merge into active batch
  │
  ▼
END ITERATION
      │
      └──────────► repeat
```

The active batch and amount of work can therefore change on every iteration.

---

## End-to-End Execution Trace

Our final engine produced:

```text
STEP 1
  decode : []
  prefill: ['A']
  finished: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 40 / 50
  scheduler: B prefill deferred (43 tokens > 10 remaining)
  KV batch: 1 | KV length: 40
```

Let's break this down.

The engine begins with:

```text
waiting = [A, B, C]

running = []
```

A's prompt contains:

```text
40 tokens
```

so its prefill costs:

```text
40 / 50
```

tokens from the iteration budget.

That leaves:

```text
10 tokens
```

B requires:

```text
43 tokens
```

for prefill.

Since:

```text
43 > 10
```

B cannot fit.

Therefore:

```text
A → prefill

B → remains waiting
C → remains waiting
```

---

### Step 2

```text
STEP 2
  decode : ['A']
  prefill: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 1 / 50
```

A has already completed prefill.

Therefore A now performs decode:

```text
A decode cost = 1 token
```

There is plenty of logical token budget remaining.

However, our current educational engine uses a **dense Transformers KV cache**.

B's prompt is physically longer than A's current cache:

```text
B prompt = 43

A cache = 41
```

so B cannot yet be merged into that dense cache.

Therefore we see:

```text
B waiting for dense-cache alignment
```

This is not a fundamental rule of continuous batching.

It is a limitation of the dense-cache representation we are using to teach the mechanism.

A production paged KV-cache system handles request memory much more flexibly.

---

### Step 3

```text
STEP 3
  decode : ['A']
  prefill: []
  running: ['A']
  waiting: ['B', 'C']
  budget used: 1 / 50
  KV length: 42
```

A decodes another token.

Its cache grows:

```text
41 → 42
```

B still requires a physical length of:

```text
43
```

so it remains waiting.

---

### Step 4: A Finishes and B Enters

```text
STEP 4
  decode : ['A']
  prefill: ['B']
  finished: ['A']
  running: ['B']
  waiting: ['C']
  budget used: 44 / 50
```

This is an important iteration.

First:

```text
A decode = 1 token
```

A reaches its stopping condition and finishes.

So A is immediately removed from the active batch.

That frees execution capacity.

B can now be admitted and prefetched:

```text
B prefill = 43 tokens
```

Total token work:

```text
A decode = 1

B prefill = 43

total = 44
```

which fits inside:

```text
max_tokens_per_step = 50
```

So the active batch changes:

```text
[A]
 ↓
[]
 ↓
[B]
```

inside the serving process.

---

### Step 5: Prefill and Decode Coexist

Our most important iteration is:

```text
STEP 5
  decode : ['B']
  prefill: ['C']
  finished: []
  running: ['B', 'C']
  waiting: []
  budget used: 39 / 50
  KV batch: 2
```

At the beginning of this step:

```text
B → already running

C → waiting
```

B already has a KV cache, so it only needs:

```text
1 decode token
```

C is new and needs:

```text
38 prefill tokens
```

Therefore:

```text
B decode  = 1

C prefill = 38

----------------

total     = 39
```

which fits inside our budget of:

```text
50
```

The scheduler performs both kinds of work during the same engine iteration.

Afterwards:

```text
running = [B, C]
```

and their KV state has been aligned into the same active batch.

This is the key scheduling situation we wanted to reach:

```text
existing request
      ↓
   DECODE
      │
      │
      ├──── same scheduler iteration
      │
      │
      ▼
new request
      ↓
   PREFILL
```

After C's prefill completes, both B and C can participate in normal batched decode.

---

### Steps 6-9: Batched Decode

The following iterations show:

```text
STEP 6
decode: ['B', 'C']
budget used: 2 / 50
```

and similarly for Steps 7, 8 and 9.

Why does each iteration use only:

```text
2 tokens
```

?

Because:

```text
B decode = 1 token
C decode = 1 token
```

So:

```text
1 + 1 = 2
```

Both requests are now running through the same batched decode operation:

```text
B token ─┐
         ├── MODEL
C token ─┘
            ↓
      next token B
      next token C
```

The KV cache also reports:

```text
KV batch: 2
```

showing that both requests are represented inside the active batched cache.

---

### Step 10: Both Requests Finish

Finally:

```text
STEP 10
  decode : ['B', 'C']
  finished: ['B', 'C']
  running: []
  waiting: []
  budget used: 2 / 50
```

Both requests reach their stopping conditions.

The scheduler removes them.

Now:

```text
waiting = []

running = []
```

so:

```python
while waiting or running:
```

becomes false and the engine terminates.

---

### Complete Request Lifecycle

Across the entire run, we observed:

```text
STEP 1
A PREFILL

        ↓

STEP 2
A DECODE

        ↓

STEP 3
A DECODE

        ↓

STEP 4
A DECODE + FINISH
B PREFILL

        ↓

STEP 5
B DECODE
C PREFILL

        ↓

STEP 6+
B + C BATCHED DECODE

        ↓

B + C FINISH
```

The active set therefore changed dynamically:

```text
[]

↓

[A]

↓

[B]

↓

[B, C]

↓

[]
```

without restarting the whole generation process.

---

### Final Generated Outputs

Our engine produced:

```text
Request A:
The capital of France
```

```text
Request B:
The largest planet in our solar system
```

```text
Request C:
Machine learning is a type of
```

The responses are intentionally short because we used small `max_new_tokens` values to make requests finish at different iterations.

The goal of this experiment is not response quality.

The goal is to observe the serving behavior.

---

## Part 5B Conclusion

We started Part 5B with only a static batched prefill.

We now have:

```text
batched prefill                         ✅

batched decode                          ✅

per-request stopping                    ✅

finished-request removal                ✅

KV-cache batch compaction               ✅

dynamic request admission               ✅

[A, B] → [B] → [B, C]                  ✅

prefill + decode coexistence            ✅

token-budget-aware scheduling           ✅

prefill/decode cost accounting          ✅

prefill deferral when budget is full    ✅

continuous active-batch changes         ✅

end-to-end serving loop                 ✅
```

The scheduler has moved from thinking only about:

```text
number of requests
```

to reasoning about:

```text
amount and type of work per iteration
```

That is a major step toward the architecture of a real LLM serving engine.

---

### Current Engine

At this point our serving loop can be summarized as:

```text
                REQUESTS
                    │
                    ▼
                WAITING
                    │
                    ▼
              ┌───────────┐
              │ SCHEDULER │
              └─────┬─────┘
                    │
              TOKEN BUDGET
                    │
           ┌────────┴────────┐
           │                 │
         PREFILL           DECODE
           │                 │
           └────────┬────────┘
                    │
                    ▼
             BATCHED MODEL
                EXECUTION
                    │
                    ▼
               KV CACHE
                    │
                    ▼
          UPDATE REQUEST STATE
                    │
          ┌─────────┴─────────┐
          │                   │
       FINISHED            RUNNING
          │                   │
          └───────────┬───────┘
                      │
                      ▼
                   repeat
```

There is one major weakness remaining.

Our KV cache is still represented as large dense tensors.

When requests enter or leave, we sometimes physically slice, align or concatenate those tensors.

That works for understanding continuous batching, but it is not how we want a serious serving engine to manage memory.

That gives us the next problem naturally:

```text
How should an inference engine actually allocate,
reuse and free KV-cache memory for many requests?
```

That is what we will solve next with **KV-cache management**.

### Part 5B Complete

**Part 5B - Dynamic Continuous Batching is now complete.**

Next is Part 6 - KV-Cache Management